<a href="https://colab.research.google.com/github/shreya1110-dev/MTechCodeFiles/blob/main/Assignments/Sem2/ACI/Emergency_Plan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1 – PS7

In [47]:
def create_test_input_files():
    # Test Case 1
    tc1_content = """START: A
GOAL: G

NODE: A 10
NODE: B 8
NODE: C 5
NODE: D 7
NODE: E 3
NODE: F 2
NODE: G 0

EDGE: A B
EDGE: A C
EDGE: B D
EDGE: B E
EDGE: C F
EDGE: D G
EDGE: E G
EDGE: F G
"""

    # Test Case 2
    tc2_content = """START: S
GOAL: G

NODE: S 9
NODE: A 7
NODE: B 4
NODE: C 6
NODE: D 2
NODE: E 1
NODE: G 0

EDGE: S A
EDGE: S B
EDGE: A C
EDGE: A D
EDGE: B E
EDGE: C G
EDGE: D G
EDGE: E G
"""

    with open("inputPSXX_TC1.txt", "w") as f1:
        f1.write(tc1_content)

    with open("inputPSXX_TC2.txt", "w") as f2:
        f2.write(tc2_content)


if __name__ == "__main__":
    create_test_input_files()
    print("Created inputPSXX_TC1.txt and inputPSXX_TC2.txt")

Created inputPSXX_TC1.txt and inputPSXX_TC2.txt


In [48]:
import heapq
import time
import os
import sys


In [49]:
class Graph:
    def __init__(self):
        self.adj = {}
        self.h = {}     # HEURISTICS

    def add_node(self, node, heuristic):
        if node not in self.adj:
            self.adj[node] = []
        self.h[node] = heuristic

    def add_edge(self, u, v):
        if u not in self.adj:
            self.adj[u] = []
        if v not in self.adj:
            self.adj[v] = []
        if v not in self.adj[u]:
            self.adj[u].append(v)
        if u not in self.adj[v]:
            self.adj[v].append(u)

    def neighbors(self, node):
        return self.adj.get(node, [])

    def heuristic(self, node):
        return self.h[node]

In [50]:
class PriorityQueue:
    def __init__(self, capacity=None):
        self.heap = []
        self.capacity = capacity

    def insert(self, node, priority):
        if self.capacity is not None and len(self.heap) >= self.capacity:
            print("Queue is full, cannot insert:", node)
            return
        heapq.heappush(self.heap, (priority, node))

    def delete(self):
        if not self.heap:
            print("Queue is empty, cannot delete")
            return None
        return heapq.heappop(self.heap)

    def is_empty(self):
        return len(self.heap) == 0

    def size(self):
        return len(self.heap)

In [51]:
def estimate_memory_bytes(frontier_heap, came_from, visited, expansion_order):
    mem = 0

    mem += sys.getsizeof(frontier_heap)
    for item in frontier_heap:
        mem += sys.getsizeof(item)
        for elem in item:
            mem += sys.getsizeof(elem)

    mem += sys.getsizeof(came_from)
    for k, v in came_from.items():
        mem += sys.getsizeof(k)
        mem += sys.getsizeof(v)

    mem += sys.getsizeof(visited)
    for elem in visited:
        mem += sys.getsizeof(elem)

    mem += sys.getsizeof(expansion_order)
    for elem in expansion_order:
        mem += sys.getsizeof(elem)

    return mem

In [52]:
def greedy_best_first_search(graph, start, goal):

    if start not in graph.adj or goal not in graph.adj:
        return None, [], 0, 0, 0

    queue = PriorityQueue()
    queue.insert(start, graph.heuristic(start))

    came_from = {start: None}
    visited = set()
    expansion_order = []

    max_queue_size = queue.size()
    nodes_expanded = 0

    max_memory_bytes = estimate_memory_bytes(queue.heap, came_from, visited, expansion_order)

    while not queue.is_empty():
        item = queue.delete()
        if item is None:
            break
        current_priority, current = item
        if current in visited:
            continue

        visited.add(current)
        expansion_order.append(current)
        nodes_expanded += 1

        if current == goal:
            break

        for nb in graph.neighbors(current):
            if nb not in visited and nb not in came_from:
                came_from[nb] = current
                queue.insert(nb, graph.heuristic(nb))

        if queue.size() > max_queue_size:
            max_queue_size = queue.size()

        mem_now = estimate_memory_bytes(queue.heap, came_from, visited, expansion_order)
        if mem_now > max_memory_bytes:
            max_memory_bytes = mem_now

    if goal not in visited:
        return None, expansion_order, nodes_expanded, max_frontier_size, max_memory_bytes

    path = []
    node = goal
    while node is not None:
        path.append(node)
        node = came_from[node]
    path.reverse()

    return path, expansion_order, nodes_expanded, max_frontier_size, max_memory_bytes


In [53]:
def read_input_file(filename):
    graph = Graph()
    start_node = None
    goal_node = None

    with open(filename, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue

            if line.startswith("START:"):
                start_node = line.split(":", 1)[1].strip()
            elif line.startswith("GOAL:"):
                goal_node = line.split(":", 1)[1].strip()
            elif line.startswith("NODE:"):
                parts = line.split(":", 1)[1].strip().split()
                if len(parts) >= 2:
                    node = parts[0]
                    h_val = float(parts[1])
                    graph.add_node(node, h_val)
            elif line.startswith("EDGE:"):
                parts = line.split(":", 1)[1].strip().split()
                if len(parts) >= 2:
                    u, v = parts[0], parts[1]
                    graph.add_edge(u, v)

    return graph, start_node, goal_node


In [54]:
def derive_output_filename(input_filename):
    base = os.path.basename(input_filename)
    name, ext = os.path.splitext(base)
    if name.startswith("input"):
        output_name = "output" + name[len("input"):] + ext
    else:
        output_name = "output_" + base
    return output_name

In [55]:
def write_output_file(filename, graph, start_node, goal_node,
                      path, expansion_order, nodes_expanded,
                      max_frontier_size, memory_used_bytes,
                      elapsed_time_ms, total_nodes):
    with open(filename, "w") as f:
        f.write("Emergency Route Planning using Greedy Best First Search\n")
        f.write("f(n) = h(n)\n\n")

        f.write("Start node: " + str(start_node) + "\n")
        f.write("Goal node: " + str(goal_node) + "\n\n")

        f.write("Visited / Expansion order:\n")
        if expansion_order:
            f.write(" -> ".join(expansion_order) + "\n\n")
        else:
            f.write("None\n\n")

        f.write("Final path from source to goal:\n")
        if path is not None:
            f.write(" -> ".join(path) + "\n")
            total_cost = len(path) - 1
            f.write("Total cost / path length: " + str(total_cost) + "\n\n")
        else:
            f.write("No path found\n\n")

        f.write("Empirical time and space usage:\n")
        f.write("Nodes expanded: " + str(nodes_expanded) + "\n")
        f.write("Maximum frontier size: " + str(max_frontier_size) + "\n")
        f.write("Space complexity (estimated memory): " + str(memory_used_bytes) + " bytes\n")
        f.write("Total nodes in graph: " + str(total_nodes) + "\n")
        f.write("Elapsed time (ms): " + str(round(elapsed_time_ms, 3)) + "\n")

In [59]:

create_test_input_files()

input_file = "inputPSXX_TC2.txt"

graph, file_start, file_goal = read_input_file(input_file)
output_file = derive_output_filename(input_file)


user_start = input(f"Enter start node (press Enter to use {file_start}): ").strip()
if user_start == "":
    start_node = file_start
else:
    start_node = user_start

goal_node = file_goal

t0 = time.time()
path, expansion_order, nodes_expanded, max_frontier_size, memory_used_bytes = greedy_best_first_search(
    graph, start_node, goal_node
)
t1 = time.time()
elapsed_time_ms = (t1 - t0) * 1000.0

total_nodes = len(graph.adj)

print("\nVisited / Expansion order:")
print(" -> ".join(expansion_order) if expansion_order else "None")

if path is not None:
    print("\nFinal path:")
    print(" -> ".join(path))
    print("Total cost / path length:", len(path) - 1)
else:
    print("\nNo path found from", start_node, "to", goal_node)

print("\nEmpirical stats:")
print("Nodes expanded:", nodes_expanded)
print("Maximum frontier size:", max_frontier_size)
print("Space complexity (estimated memory):", memory_used_bytes, "bytes")
print("Total nodes in graph:", total_nodes)
print("Elapsed time (ms):", round(elapsed_time_ms, 3))

write_output_file(
    output_file,
    graph,
    start_node,
    goal_node,
    path,
    expansion_order,
    nodes_expanded,
    max_frontier_size,
    memory_used_bytes,
    elapsed_time_ms,
    total_nodes,
)
print("\nResults written to:", output_file)

Enter start node (press Enter to use S): 

Visited / Expansion order:
S -> B -> E -> G

Final path:
S -> B -> E -> G
Total cost / path length: 3

Empirical stats:
Nodes expanded: 4
Maximum frontier size: 2
Space complexity (estimated memory): 1466 bytes
Total nodes in graph: 7
Elapsed time (ms): 0.244

Results written to: outputPSXX_TC2.txt
